# pipe_catedra/01 — Preprocesamiento (porteo de `preprocesamiento_1.ipynb` / z301)

Input  : `sell-in.txt.gz`, `tb_productos.txt`, `product_id_apredecir201912.txt`
Output : `z301_preprocessed_{modo}.parquet`

Porteo fiel del notebook de la catedra (misma logica, mismos nombres de
columnas y de PARAM) -- unico cambio real: los dos pasos que en el
original recorren productos/periodos con loops de Python (calculo de
`productos_nuevos_*_3m` y el armado de la grilla que completa ceros entre
nacimiento y muerte) se reescriben como consultas DuckDB (join +
`BETWEEN` sobre una tabla de indices de periodo), que hacen exactamente
lo mismo pero en una sola pasada vectorizada en vez de un loop fila por
fila -- son los dos puntos mas lentos del notebook original.

Responsabilidades (identicas al original):
- Completar ceros entre fecha de nacimiento y fecha de muerte de cada producto
- Crear columna `agrupa_id` (producto o cliente_producto)
- Join con `tb_productos` para agregar jerarquia de categorias
- Features de categoria (totales, activos, canibalizacion) sobre el universo COMPLETO
- Features de estructura de clientes (clientes activos, HHI, cliente principal)
- Switch para usar solo los 780 productos a predecir o todos


## 0) Setup


In [ ]:
import os, shutil, subprocess, time
from pathlib import Path

import polars as pl
import numpy as np
import duckdb


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_RAW = BUCKET / "datasets"
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_RAW.mkdir(parents=True, exist_ok=True)
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"BUCKET : {BUCKET}")
print(f"crudos : {DIR_RAW}")
print(f"salida : {DIR_OUT}")

# Kaggle auth: usar el de ~/.kaggle si ya existe; si no, buscarlo en el bucket
kaggle_dst = Path.home() / ".kaggle" / "kaggle.json"
kaggle_dst.parent.mkdir(parents=True, exist_ok=True)
if kaggle_dst.exists():
    kaggle_dst.chmod(0o600)
    print("Kaggle auth OK (ya estaba en ~/.kaggle)")
else:
    _encontrado = False
    for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
        if cand.exists():
            shutil.copy(cand, kaggle_dst)
            kaggle_dst.chmod(0o600)
            print(f"Kaggle auth OK (copiado de {cand})")
            _encontrado = True
            break
    if not _encontrado:
        print("aviso: kaggle.json no encontrado. Subilo a ~/.kaggle/kaggle.json o al bucket.")


def descargar(archivo):
    url = f"https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/{archivo}"
    dst = DIR_RAW / archivo
    if not dst.exists():
        subprocess.run(["wget", url, "-O", str(dst)], check=True)
    print(f"ok: {archivo}")


for _a in ("sell-in.txt.gz", "tb_productos.txt", "tb_stocks.txt", "product_id_apredecir201912.txt"):
    if (DIR_RAW / _a).exists():
        print(f"ya existe: {_a}")
    else:
        descargar(_a)


## 1) Parametros


In [ ]:
PARAM = {
    'experimento': 'z301',

    # PALANCA 1: granularidad
    # 'producto'         -> agrupa_id = product_id
    # 'cliente_producto' -> agrupa_id = customer_id * 10000 + product_id
    'modo_agrupacion': 'producto',

    # PALANCA 2: universo de productos
    'solo_predecir': True,

    'path_sellin':    str(DIR_RAW / 'sell-in.txt.gz'),
    'path_productos': str(DIR_RAW / 'tb_productos.txt'),
    'path_apredecir': str(DIR_RAW / 'product_id_apredecir201912.txt'),
}

# El nombre incluye solo_predecir ademas del modo -- si no, cambiar esta
# palanca sin tocar modo_agrupacion reusaria en silencio el parquet de la
# corrida anterior con el universo de productos equivocado (bug real,
# encontrado corriendo esta version en la VM).
_tag_solo = 'solo780' if PARAM['solo_predecir'] else 'todosProductos'
PARAM['path_output'] = str(DIR_OUT / f"z301_preprocessed_{PARAM['modo_agrupacion']}_{_tag_solo}.parquet")

print('Parametros:', PARAM)
print('Output:', PARAM['path_output'])


## 2) Carga de datos


In [ ]:
df_raw = pl.read_csv(
    PARAM['path_sellin'],
    separator='\t',
    schema_overrides={
        'periodo':           pl.Int32,
        'customer_id':       pl.Int32,
        'product_id':        pl.Int32,
        'cust_request_qty':  pl.Int32,
        'cust_request_tn':   pl.Float32,
        'tn':                pl.Float32,
    }
)
print(f'Sell-in crudo: {df_raw.shape}')

tb_apredecir = pl.read_csv(
    PARAM['path_apredecir'], separator='\t',
    schema_overrides={'product_id': pl.Int32}
)
print(f'Productos a predecir: {tb_apredecir.height}')

tb_productos = pl.read_csv(PARAM['path_productos'], separator='\t')
print(f'Catalogo de productos: {tb_productos.shape}')
print('Columnas:', tb_productos.columns)
tb_productos.head(3)


## 2b) Features de categoria sobre el universo COMPLETO

Se calculan con **todos** los productos (antes de filtrar a los 780), para que los totales de categoria y la competencia incluyan productos fuera de la lista a predecir. Capturan el efecto de canibalizacion: cuando nace un producto nuevo en una categoria, los demas se ven afectados.

- `tn_cat2_total`: toneladas totales de la categoria por periodo
- `productos_activos_cat2`: cantidad de productos activos en la categoria
- `productos_nuevos_cat2_3m`: productos nacidos en los ultimos 3 meses en la categoria

**Cambio vs el original**: `productos_nuevos_*_3m` se calculaba con un loop de Python
(por cada valor de categoria, por cada periodo, contar nacimientos en la ventana) --
acá se calcula con una consulta DuckDB (join + `BETWEEN` sobre el indice ordinal de
periodo, con `IS NOT DISTINCT FROM` para que la categoria `NULL`, huerfanos del
catalogo, tambien matchee), misma semantica exacta (ventana = `[idx-2, idx]` sobre la
lista ORDENADA de periodos, no un rango calendario), una sola pasada vectorizada en
vez de recorrer cada combinacion valor x periodo en Python puro.


In [ ]:
# df_raw aca todavia es el sell-in COMPLETO (el filtro a 780 viene despues)
prod_cat = tb_productos.select(['product_id', 'cat1', 'cat2', 'cat3', 'brand']).with_columns(
    pl.col('product_id').cast(pl.Int32)
)

# Sell-in completo agregado a producto x periodo, con su jerarquia
df_univ = (
    df_raw
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('tn').sum().alias('tn'))
    .join(prod_cat, on='product_id', how='left')
)

# Indice ordinal de periodos para ventanas de N meses
periodos_u = sorted(df_univ['periodo'].unique().to_list())
p2i = {p: i for i, p in enumerate(periodos_u)}
periodos_idx = pl.DataFrame({'periodo': periodos_u, 'idx': list(range(len(periodos_u)))}) \
                 .with_columns(pl.col('periodo').cast(pl.Int32), pl.col('idx').cast(pl.Int32))

con = duckdb.connect()
con.register('df_univ', df_univ)
con.register('periodos_idx', periodos_idx)

# 1) Totales jerarquicos por nivel x periodo (universo completo)
totales = {}
for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
    totales[nivel] = (
        df_univ.group_by([nivel, 'periodo'])
               .agg(pl.col('tn').sum().alias(f'tn_total_{nivel}'))
    )

# 2) Productos activos por cat2 y cat3 x periodo
activos = {}
for nivel in ['cat2', 'cat3']:
    activos[nivel] = (
        df_univ.filter(pl.col('tn') > 0)
               .group_by([nivel, 'periodo'])
               .agg(pl.col('product_id').n_unique().alias(f'productos_activos_{nivel}'))
    )

# 3) Nacimientos recientes (canibalizacion) por cat2 y cat3
# nacimiento = primer periodo con tn>0 de cada producto
nac = (
    df_univ.filter(pl.col('tn') > 0)
           .group_by('product_id')
           .agg(pl.col('periodo').min().alias('nacimiento'))
           .join(prod_cat, on='product_id', how='left')
)
con.register('nac', nac)


def nacimientos_por_nivel(nivel):
    """Cuenta productos nacidos en la ventana [idx-2 ... idx] por nivel x periodo
    (idx = posicion en la lista ordenada de periodos, igual que el original).
    El original recorre CADA valor de nivel x TODOS los periodos (cross join),
    no solo los pares (nivel, periodo) realmente observados en df_univ -- por
    eso aca tambien es un CROSS JOIN de valores x periodos_idx, no un DISTINCT
    de pares. IS NOT DISTINCT FROM matchea NULL=NULL (productos huerfanos sin
    categoria)."""
    q = f"""
        SELECT v."{nivel}" AS "{nivel}", pi.periodo,
               COUNT(n.product_id) AS "productos_nuevos_{nivel}_3m"
        FROM (SELECT DISTINCT "{nivel}" FROM df_univ) v
        CROSS JOIN periodos_idx pi
        LEFT JOIN (
            SELECT nac.product_id, nac."{nivel}" AS "{nivel}", pin.idx AS nac_idx
            FROM nac
            JOIN periodos_idx pin ON pin.periodo = nac.nacimiento
        ) n
          ON n."{nivel}" IS NOT DISTINCT FROM v."{nivel}"
         AND n.nac_idx BETWEEN pi.idx - 2 AND pi.idx
        GROUP BY v."{nivel}", pi.periodo
    """
    return con.sql(q).pl().with_columns(pl.col('periodo').cast(pl.Int32))


t0 = time.time()
nuevos_cat2 = nacimientos_por_nivel('cat2')
nuevos_cat3 = nacimientos_por_nivel('cat3')
print(f'nacimientos por nivel (DuckDB): [{time.time()-t0:.1f}s]')

print('Features de categoria (universo completo) calculadas:')
print('  totales:', list(totales.keys()))
print('  activos:', list(activos.keys()))
print('  canibalizacion: cat2, cat3')


## 3) Filtrado y construccion de AGRUPA_ID


In [ ]:
if PARAM['solo_predecir']:
    df_raw = df_raw.join(tb_apredecir, on='product_id', how='inner')
    print(f'Filtrado a 780 productos: {df_raw.shape}')
else:
    print(f'Usando todos los productos: {df_raw.shape}')

if PARAM['modo_agrupacion'] == 'producto':
    df_agrupado = (
        df_raw
        .group_by(['product_id', 'periodo'])
        .agg(pl.col('tn').sum().alias('tn'))
        .with_columns(
            pl.col('product_id').cast(pl.Int64).alias('agrupa_id'),
            pl.lit(None).cast(pl.Int32).alias('customer_id')
        )
    )
elif PARAM['modo_agrupacion'] == 'cliente_producto':
    # MULTIPLICADOR debe ser > max(product_id) para que la codificacion
    # customer_id * MULTIPLICADOR + product_id sea inequivoca (sin colisiones).
    # product_id llega a ~21299 -> 100_000 da margen amplio.
    MULTIPLICADOR_AGRUPA = 100_000
    df_agrupado = (
        df_raw
        .group_by(['customer_id', 'product_id', 'periodo'])
        .agg(pl.col('tn').sum().alias('tn'))
        .with_columns(
            (pl.col('customer_id').cast(pl.Int64) * MULTIPLICADOR_AGRUPA
             + pl.col('product_id').cast(pl.Int64)).alias('agrupa_id')
        )
    )
else:
    raise ValueError(f"modo_agrupacion invalido: {PARAM['modo_agrupacion']}")

df_agrupado = df_agrupado.sort(['agrupa_id', 'periodo'])
print(f'Filas agrupadas: {df_agrupado.shape}')
print(f'IDs unicos: {df_agrupado["agrupa_id"].n_unique()}')


## 4) Completar ceros entre nacimiento y muerte de cada producto

Reglas (identicas al original):
- **Nacimiento**: primer periodo con `tn > 0`
- **Muerte**: ultimo periodo con `tn > 0`
- Si el ultimo periodo activo es el ultimo del dataset (201912), NO inferimos que murio
- Ceros se completan solo dentro del rango [nacimiento, muerte], ambos inclusive

**Cambio vs el original**: la grilla se armaba con un loop de Python que iteraba
`agrupa_id` por `agrupa_id` haciendo `.append()` fila por fila. Aca se arma con una
consulta DuckDB (join contra `periodos_idx` con `BETWEEN idx_inicio AND idx_fin`) --
el clasico patron de "explotar un rango via join", que hace lo mismo en una sola
pasada vectorizada en vez de un loop en Python puro (el paso mas lento del notebook
original cuando hay muchos productos con historia larga).


In [ ]:
# Todos los periodos del dataset
periodos_all = sorted(df_agrupado['periodo'].unique().to_list())
ULTIMO_PERIODO_DATASET = max(periodos_all)
print(f'Periodos: {min(periodos_all)} -> {ULTIMO_PERIODO_DATASET}  ({len(periodos_all)} meses)')

periodo_a_idx = {p: i for i, p in enumerate(periodos_all)}
idx_a_periodo = {i: p for p, i in periodo_a_idx.items()}
periodos_all_idx = pl.DataFrame({'periodo': periodos_all, 'idx': list(range(len(periodos_all)))}) \
                     .with_columns(pl.col('periodo').cast(pl.Int32), pl.col('idx').cast(pl.Int32))

# Nacimiento y muerte por agrupa_id
ventas_positivas = df_agrupado.filter(pl.col('tn') > 0)

primeros = (
    ventas_positivas
    .group_by('agrupa_id')
    .agg(pl.col('periodo').min().alias('primer_periodo_activo'))
)

ultimos = (
    ventas_positivas
    .group_by('agrupa_id')
    .agg(pl.col('periodo').max().alias('ultimo_periodo_activo'))
)

primeros_ultimos = primeros.join(ultimos, on='agrupa_id', how='left')
print(f'Productos con historia: {primeros_ultimos.height}')

# Cuantos productos estan activos hasta el ultimo periodo
n_activos = primeros_ultimos.filter(
    pl.col('ultimo_periodo_activo') == ULTIMO_PERIODO_DATASET
).height
print(f'Productos activos en {ULTIMO_PERIODO_DATASET}: {n_activos}')
print(f'Productos discontinuados antes de {ULTIMO_PERIODO_DATASET}: {primeros_ultimos.height - n_activos}')


In [ ]:
# Grid: agrupa_id x periodos entre nacimiento y muerte (via DuckDB, sin loop de Python)
con.register('primeros_ultimos', primeros_ultimos)
con.register('periodos_all_idx', periodos_all_idx)

t0 = time.time()
grid = con.sql("""
    SELECT pu.agrupa_id, pi.periodo
    FROM primeros_ultimos pu
    JOIN periodos_all_idx pi_ini ON pi_ini.periodo = pu.primer_periodo_activo
    JOIN periodos_all_idx pi_fin ON pi_fin.periodo = pu.ultimo_periodo_activo
    JOIN periodos_all_idx pi ON pi.idx BETWEEN pi_ini.idx AND pi_fin.idx
""").pl().with_columns([
    pl.col('agrupa_id').cast(pl.Int64),
    pl.col('periodo').cast(pl.Int32),
])
print(f'Grid (agrupa_id x periodo, DuckDB): {grid.shape}   [{time.time()-t0:.1f}s]')

# Join con ventas reales; faltantes -> 0
df_full = (
    grid
    .join(
        df_agrupado.select(['agrupa_id', 'periodo', 'product_id', 'customer_id', 'tn']),
        on=['agrupa_id', 'periodo'],
        how='left'
    )
    .with_columns(pl.col('tn').fill_null(0.0))
    .sort(['agrupa_id', 'periodo'])
)

print(f'Dataset con ceros completados: {df_full.shape}')
print(f'Nulos en tn: {df_full["tn"].is_null().sum()}')


## 5) Reconstruir product_id en modo producto


In [ ]:
if PARAM['modo_agrupacion'] == 'producto':
    df_full = df_full.with_columns(
        pl.col('agrupa_id').cast(pl.Int32).alias('product_id')
    )
elif PARAM['modo_agrupacion'] == 'cliente_producto':
    # Decodificar customer_id y product_id desde agrupa_id (necesario para las filas
    # de ceros completados, que vienen de la grid y no traen estas columnas).
    MULTIPLICADOR_AGRUPA = 100_000
    df_full = df_full.with_columns([
        (pl.col('agrupa_id') // MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('customer_id'),
        (pl.col('agrupa_id') % MULTIPLICADOR_AGRUPA).cast(pl.Int32).alias('product_id'),
    ])

print('Schema:', df_full.schema)
print(f'product_id nulos: {df_full["product_id"].is_null().sum()}')
print(f'customer_id nulos: {df_full["customer_id"].is_null().sum()}')


## 6) Join con tb_productos — jerarquia de categorias

Agrega `cat1`, `cat2`, `cat3`, `brand`, `sku_size` como features para LGBM.


In [ ]:
# Asegurar que product_id tiene el mismo tipo en ambas tablas
tb_productos = tb_productos.with_columns(
    pl.col('product_id').cast(pl.Int32)
)

print('Columnas de tb_productos:', tb_productos.columns)

df_full = df_full.join(tb_productos, on='product_id', how='left')

print(f'Despues del join con tb_productos: {df_full.shape}')
print(f'Nulos en cat2: {df_full["cat2"].is_null().sum() if "cat2" in df_full.columns else "cat2 no encontrado"}')
df_full.head(3)


## 6b) Features de estructura de clientes

Calculadas desde el sell-in crudo (tiene `customer_id`) por `product_id x periodo`:
- `clientes_activos`: cantidad de clientes con tn > 0
- `hhi_clientes`: concentracion Herfindahl (suma de shares al cuadrado)
- `cliente_principal_share`: share del cliente mayor

Son features del periodo actual (sin leakage, igual que lag_0).


In [ ]:
# df_raw todavia tiene customer_id (es el sell-in filtrado, antes de agregar por producto)
df_cust = (
    df_raw
    .group_by(['product_id', 'customer_id', 'periodo'])
    .agg(pl.col('tn').sum().alias('tn_cust'))
    .filter(pl.col('tn_cust') > 0)
)

df_prod_tot = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('tn_cust').sum().alias('tn_prod_tot'))
)

df_n_cust = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg(pl.col('customer_id').n_unique().alias('clientes_activos'))
)

df_cust = df_cust.join(df_prod_tot, on=['product_id', 'periodo'], how='left')
df_cust = df_cust.with_columns(
    (pl.col('tn_cust') / pl.col('tn_prod_tot')).alias('share')
)
df_conc = (
    df_cust
    .group_by(['product_id', 'periodo'])
    .agg([
        pl.col('share').max().alias('cliente_principal_share'),
        (pl.col('share') ** 2).sum().alias('hhi_clientes'),
    ])
)

df_cust_feats = df_n_cust.join(df_conc, on=['product_id', 'periodo'], how='left')
print(f'Features de clientes calculadas: {df_cust_feats.shape}')

df_cust_feats = df_cust_feats.with_columns([
    pl.col('clientes_activos').cast(pl.Int32),
    pl.col('cliente_principal_share').cast(pl.Float32),
    pl.col('hhi_clientes').cast(pl.Float32),
])

df_full = df_full.join(df_cust_feats, on=['product_id', 'periodo'], how='left')

df_full = df_full.with_columns([
    pl.col('clientes_activos').fill_null(0),
    pl.col('cliente_principal_share').fill_null(0.0),
    pl.col('hhi_clientes').fill_null(0.0),
])

_prod_muestra = df_full['product_id'][0]
print(f'Producto {_prod_muestra} (muestra):')
print(df_full.filter(pl.col('product_id') == _prod_muestra).select(
    ['periodo', 'tn', 'clientes_activos', 'cliente_principal_share', 'hhi_clientes']
).head(5))


## 6c) Unir features de categoria y calcular market share

Se unen las features de categoria (calculadas sobre el universo completo) y se calcula `market_share = tn / tn_cat2_total` con el total correcto.


In [ ]:
for nivel in ['cat1', 'cat2', 'cat3', 'brand']:
    df_full = df_full.join(totales[nivel], on=[nivel, 'periodo'], how='left')

for nivel in ['cat2', 'cat3']:
    df_full = df_full.join(activos[nivel], on=[nivel, 'periodo'], how='left')

df_full = df_full.join(nuevos_cat2, on=['cat2', 'periodo'], how='left')
df_full = df_full.join(nuevos_cat3, on=['cat3', 'periodo'], how='left')

df_full = df_full.with_columns([
    pl.col('tn_total_cat1').fill_null(0.0),
    pl.col('tn_total_cat2').fill_null(0.0),
    pl.col('tn_total_cat3').fill_null(0.0),
    pl.col('tn_total_brand').fill_null(0.0),
    pl.col('productos_activos_cat2').fill_null(0),
    pl.col('productos_activos_cat3').fill_null(0),
    pl.col('productos_nuevos_cat2_3m').fill_null(0),
    pl.col('productos_nuevos_cat3_3m').fill_null(0),
])

df_full = df_full.with_columns([
    pl.when(pl.col('tn_total_cat2') > 0).then(pl.col('tn') / pl.col('tn_total_cat2'))
      .otherwise(pl.lit(0.0)).alias('market_share_cat2'),
    pl.when(pl.col('tn_total_cat3') > 0).then(pl.col('tn') / pl.col('tn_total_cat3'))
      .otherwise(pl.lit(0.0)).alias('market_share_cat3'),
    pl.when(pl.col('tn_total_brand') > 0).then(pl.col('tn') / pl.col('tn_total_brand'))
      .otherwise(pl.lit(0.0)).alias('market_share_brand'),
])
# Mantener 'market_share' (alias del de cat2) por compatibilidad con experimentos previos
df_full = df_full.with_columns(pl.col('market_share_cat2').alias('market_share'))

print('Features jerarquicas unidas. Muestra producto', _prod_muestra, ':')
print(df_full.filter(pl.col('product_id') == _prod_muestra).select(
    ['periodo', 'tn', 'tn_total_cat2', 'tn_total_cat3', 'market_share_cat2',
     'market_share_cat3', 'productos_nuevos_cat2_3m', 'productos_nuevos_cat3_3m']
).head(5))


## 7) Metadatos y verificacion


In [ ]:
df_full = df_full.with_columns([
    pl.lit(PARAM['modo_agrupacion']).alias('modo_agrupacion'),
    pl.lit(PARAM['solo_predecir']).alias('solo_predecir'),
])

print(f'Producto {_prod_muestra}:')
print(df_full.filter(pl.col('product_id') == _prod_muestra).select(
    ['product_id', 'periodo', 'tn', 'cat1', 'cat2', 'brand']
).head(5))


## 8) Guardar output


In [ ]:
df_full.write_parquet(PARAM['path_output'])
print(f'Guardado: {PARAM["path_output"]}')
print(f'   Filas: {df_full.height:,}')
print(f'   Columnas: {df_full.columns}')

df_check = pl.read_parquet(PARAM['path_output'])
print(f'   Lectura OK: {df_check.shape}')
